<a href="https://colab.research.google.com/github/aluminium5/Hack-O-Week-Sem-V-2026-27/blob/main/week15_week16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Hack-o-Week — Weeks 15 & 16: Model Deployment Basics & Neural Networks from Scratch

Welcome to **Weeks 15 & 16** of Hack-o-Week! In previous weeks, we explored data preparation, linear algebra, supervised regression/classification, clustering, dimensionality reduction, and ensemble learning.

Now, we bridge two critical pillars of modern data science:
1. **Model Deployment Basics**: Moving beyond experimental Jupyter notebooks by serializing, persisting, and reloading production-ready models using Python's `pickle` and `joblib`.
2. **Foundations of Deep Learning from Scratch**: Peeling back high-level abstraction frameworks (like PyTorch or TensorFlow) to build an end-to-end multi-layer neural network using **only pure NumPy**.

---

### Learning Objectives
By the end of this practical lab, you will:
- Understand model persistence, state serialization, and trade-offs between `pickle` and `joblib`.
- Build an end-to-end deployment pipeline with preprocessors and test inference with JSON payloads.
- Understand the biological and mathematical Perceptron and why the XOR problem exposed single-layer limitations.
- Explore activation functions ($\sigma$, $	anh$, $	ext{ReLU}$, $	ext{Leaky ReLU}$, $	ext{Softmax}$), their derivatives, and the vanishing gradient problem.
- Formulate loss functions (Binary Cross-Entropy & MSE) and optimization dynamics ($	ext{SGD}$, $	ext{Momentum}$, and $	ext{Adam}$).
- Derive and implement forward propagation, backward propagation, and gradient updates using vector calculus.
- Train your custom NumPy neural net on non-linear datasets, plot decision boundaries, and serialize the trained neural network.


## 0. Environment Setup & Imports

Let's import our foundational packages. We will use `numpy` for all mathematical matrix operations, `matplotlib` and `seaborn` for visualization, and standard library `pickle` alongside `joblib` for deployment serialization.


In [ ]:
import os
import sys
import time
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

print("All dependencies imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Joblib version: {joblib.__version__}")


## 1. Intro to Model Deployment Basics

### What is Model Deployment?
Training a machine learning model inside a Jupyter notebook produces an in-memory object (weights, decision tree splits, scaling statistics). Once the notebook session terminates, all learned parameters are lost.

**Deployment** is the process of taking that trained model and integrating it into a production environment where it can accept new, unseen queries and deliver real-time or batch predictions.

```
+-------------------+      Serialize       +--------------------+      Load & Serve      +-----------------------+
|  Trained Model    |  ----------------->  |   Artifact File    |  ------------------->  |  Inference Service    |
| (in-memory state) |  (pickle / joblib)   | (.pkl / .joblib)   |  (FastAPI/Flask API)   | (Predict on real data)|
+-------------------+                      +--------------------+                        +-----------------------+
```

### The Serialization Mechanism: `pickle` vs. `joblib`
1. **`pickle`**: Python's native object serialization protocol. Converts nearly arbitrary Python objects into a byte stream.
2. **`joblib`**: Specifically optimized for objects that carry large internal **NumPy arrays** (e.g. Scikit-learn estimators). It provides efficient disk-spilling, multi-file storage, and built-in compression (`zlib`, `gzip`, `lz4`).

Let's prepare a student performance classification dataset to train and serialize our first deployment-ready model.


In [ ]:
# Dataset generator consistent with previous weeks
def get_student_dataset(n_samples=1200):
    np.random.seed(RANDOM_STATE)
    study_hours = np.clip(np.random.normal(4.5, 1.8, n_samples), 0, 12)
    sleep_hours = np.clip(np.random.normal(7.0, 1.1, n_samples), 3, 10)
    attendance = np.clip(np.random.normal(82, 12, n_samples), 30, 100)
    social_media = np.clip(np.random.normal(3.0, 1.5, n_samples), 0, 9)
    quiz_score = np.clip(np.random.normal(70, 15, n_samples), 10, 100)
    
    # Latent score determining pass (1) or fail (0)
    latent_score = (
        study_hours * 5.0 + 
        attendance * 0.35 + 
        quiz_score * 0.45 - 
        social_media * 3.5 + 
        sleep_hours * 1.5 + 
        np.random.normal(0, 8, n_samples)
    )
    passed = (latent_score >= 82).astype(int)
    
    df = pd.DataFrame({
        "study_hours": np.round(study_hours, 1),
        "sleep_hours": np.round(sleep_hours, 1),
        "attendance_pct": np.round(attendance, 1),
        "social_media_hours": np.round(social_media, 1),
        "quiz_score": np.round(quiz_score, 1),
        "passed": passed
    })
    return df

df_students = get_student_dataset()
print("Dataset preview:")
display(df_students.head())
print(f"Total samples: {len(df_students)} | Pass rate: {df_students['passed'].mean():.2%}")


### 1.1 Packaging Preprocessing and Model into a Pipeline
A common production anti-pattern is saving *only* the model weights while forgetting the feature transformations (e.g. mean and standard deviation from `StandardScaler`). When raw data arrives in production, unscaled inputs result in catastrophic prediction failures.

Always package your **data transformation + model** together into an atomic `Pipeline`!


In [ ]:
features = ["study_hours", "sleep_hours", "attendance_pct", "social_media_hours", "quiz_score"]
X = df_students[features]
y = df_students["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Atomic pipeline: Scaler + Classifier
production_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE))
])

production_pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, production_pipeline.predict(X_train))
test_acc = accuracy_score(y_test, production_pipeline.predict(X_test))

print(f"Model Training Accuracy: {train_acc:.4f}")
print(f"Model Testing Accuracy:  {test_acc:.4f}")


### 1.2 Serialization Benchmark: `pickle` vs `joblib`
Now let's save the pipeline using both libraries, measure file sizes, execution times, and test round-trip fidelity.


In [ ]:
os.makedirs("models", exist_ok=True)
pickle_path = "models/pipeline_v1.pkl"
joblib_path = "models/pipeline_v1.joblib"
joblib_compressed_path = "models/pipeline_v1_compressed.joblib"

# 1. Pickle Dump
t0 = time.time()
with open(pickle_path, "wb") as f:
    pickle.dump(production_pipeline, f)
pickle_dump_time = (time.time() - t0) * 1000

# 2. Joblib Dump (default)
t0 = time.time()
joblib.dump(production_pipeline, joblib_path)
joblib_dump_time = (time.time() - t0) * 1000

# 3. Joblib Dump (with compression level 3)
joblib.dump(production_pipeline, joblib_compressed_path, compress=3)

# 4. Measure Load Times
t0 = time.time()
with open(pickle_path, "rb") as f:
    loaded_pickle_model = pickle.load(f)
pickle_load_time = (time.time() - t0) * 1000

t0 = time.time()
loaded_joblib_model = joblib.load(joblib_path)
joblib_load_time = (time.time() - t0) * 1000

# Compare file sizes
size_pickle = os.path.getsize(pickle_path) / 1024
size_joblib = os.path.getsize(joblib_path) / 1024
size_compressed = os.path.getsize(joblib_compressed_path) / 1024

benchmark_df = pd.DataFrame([
    {"Method": "pickle (.pkl)", "File Size (KB)": f"{size_pickle:.1f}", "Dump Time (ms)": f"{pickle_dump_time:.2f}", "Load Time (ms)": f"{pickle_load_time:.2f}"},
    {"Method": "joblib (.joblib)", "File Size (KB)": f"{size_joblib:.1f}", "Dump Time (ms)": f"{joblib_dump_time:.2f}", "Load Time (ms)": f"{joblib_load_time:.2f}"},
    {"Method": "joblib compressed", "File Size (KB)": f"{size_compressed:.1f}", "Dump Time (ms)": "-", "Load Time (ms)": "-"}
])
display(benchmark_df)

# Assert strict mathematical equivalence
orig_preds = production_pipeline.predict_proba(X_test)
pickle_preds = loaded_pickle_model.predict_proba(X_test)
joblib_preds = loaded_joblib_model.predict_proba(X_test)

assert np.allclose(orig_preds, pickle_preds), "Pickle predictions do not match original model!"
assert np.allclose(orig_preds, joblib_preds), "Joblib predictions do not match original model!"
print("Verification Passed: 100% bitwise prediction match between in-memory and deserialized models!")


### 1.3 Simulating an Online Production Inference Endpoint
In a web service (e.g. FastAPI, Flask, or AWS Lambda), an HTTP `POST` request provides a JSON payload. The deployment function unpacks this dictionary into a DataFrame, runs pipeline inference, and returns probabilities.


In [ ]:
def predict_student_status(raw_json_payload: dict, model_artifact_path: str = "models/pipeline_v1.joblib") -> dict:
    """
    Simulates a production microservice endpoint for inference.
    """
    # 1. Load model artifact (in production this is cached at app startup)
    deployed_model = joblib.load(model_artifact_path)
    
    # 2. Parse incoming JSON to DataFrame
    input_df = pd.DataFrame([raw_json_payload])
    
    # 3. Predict probability and class label
    prob_pass = deployed_model.predict_proba(input_df)[0][1]
    prediction = int(prob_pass >= 0.5)
    
    return {
        "status": "success",
        "predicted_class": "PASS" if prediction == 1 else "FAIL",
        "pass_probability": round(float(prob_pass), 4),
        "risk_level": "LOW" if prob_pass >= 0.75 else ("MEDIUM" if prob_pass >= 0.5 else "HIGH")
    }

# Test sample query payload (e.g., sent from a mobile app or student portal)
sample_student_payload = {
    "study_hours": 5.5,
    "sleep_hours": 7.5,
    "attendance_pct": 92.0,
    "social_media_hours": 1.5,
    "quiz_score": 85.0
}

response = predict_student_status(sample_student_payload)
print("Production API Response:")
for k, v in response.items():
    print(f"  {k}: {v}")


---
## 2. Perceptrons, Activation Functions & The Non-Linearity Problem

Now that we understand how models are deployed, let us understand how deep neural architectures are constructed from the ground up.

### 2.1 The Perceptron: The Fundamental Building Block
Proposed by Frank Rosenblatt (1958), the **Perceptron** computes a linear combination of its inputs $\mathbf{x} = [x_1, x_2, \dots, x_d]^T$ weighted by $\mathbf{w} = [w_1, w_2, \dots, w_d]^T$, adds a scalar bias $b$, and applies an activation function:

$$z = \mathbf{w}^T \mathbf{x} + b = \sum_{i=1}^d w_i x_i + b$$
$$\hat{y} = \sigma(z)$$

Let's implement a single perceptron from scratch in NumPy:


In [ ]:
class NumPyPerceptron:
    def __init__(self, learning_rate=0.1, n_epochs=50):
        self.lr = learning_rate
        self.n_epochs = n_epochs
        self.weights = None
        self.bias = None
        self.errors_history = []
        
    def step_function(self, z):
        return np.where(z >= 0, 1, 0)
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        
        for epoch in range(self.n_epochs):
            errors = 0
            for xi, target in zip(X, y):
                linear_output = np.dot(xi, self.weights) + self.bias
                y_predicted = self.step_function(linear_output)
                update = self.lr * (target - y_predicted)
                self.weights += update * xi
                self.bias += update
                errors += int(update != 0.0)
            self.errors_history.append(errors)
            if errors == 0:
                break
        return self
        
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self.step_function(linear_output)


### 2.2 The Linearity Limit: The Historic XOR Problem (Minsky & Papert, 1969)
A single perceptron can easily learn linearly separable logic gates like **AND** and **OR**. However, it fundamentally cannot solve the **XOR (Exclusive OR)** gate, because XOR requires a non-linear decision boundary.

Let's demonstrate this failure visually:


In [ ]:
# Logic Gate Truth Tables
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])
y_or  = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])

p_and = NumPyPerceptron().fit(X_logic, y_and)
p_or  = NumPyPerceptron().fit(X_logic, y_or)
p_xor = NumPyPerceptron().fit(X_logic, y_xor)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

gates = [("AND Gate (Linearly Separable)", p_and, y_and),
         ("OR Gate (Linearly Separable)", p_or, y_or),
         ("XOR Gate (NOT Linearly Separable)", p_xor, y_xor)]

for ax, (title, model, y_true) in zip(axes, gates):
    # Scatter points
    ax.scatter(X_logic[y_true==0, 0], X_logic[y_true==0, 1], color="crimson", s=120, edgecolors="k", label="Class 0")
    ax.scatter(X_logic[y_true==1, 0], X_logic[y_true==1, 1], color="dodgerblue", s=120, edgecolors="k", label="Class 1")
    
    # Plot decision boundary: w1*x1 + w2*x2 + b = 0 => x2 = -(w1*x1 + b) / w2
    x1_vals = np.linspace(-0.5, 1.5, 100)
    if model.weights[1] != 0:
        x2_vals = -(model.weights[0] * x1_vals + model.bias) / model.weights[1]
        ax.plot(x1_vals, x2_vals, "--k", label="Decision Boundary")
    
    acc = accuracy_score(y_true, model.predict(X_logic))
    ax.set_title(f"{title}\nAccuracy: {acc:.0%}", fontsize=12)
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.legend(loc="lower left")

plt.tight_layout()
plt.show()


**Key Takeaway**: To learn complex, non-linear functions (like XOR and real-world data patterns), we must:
1. Stack multiple perceptrons into **hidden layers** (Multi-Layer Perceptron / MLP).
2. Replace simple step functions with **continuous, differentiable non-linear activation functions**.

---

### 2.3 Activation Functions & Their Derivatives
Activation functions introduce non-linearity into network layers. Without them, stacking 100 linear layers would still collapse mathematically into a single linear regression!

Here are the key activation functions, their formulas, and their first derivatives:

| Activation | Equation $a = g(z)$ | Derivative $g'(z) = rac{da}{dz}$ | Primary Use Case |
| :--- | :--- | :--- | :--- |
| **Sigmoid** | $\sigma(z) = rac{1}{1 + e^{-z}}$ | $\sigma(z)(1 - \sigma(z))$ | Binary classification output |
| **Tanh** | $	anh(z) = rac{e^z - e^{-z}}{e^z + e^{-z}}$ | $1 - 	anh^2(z)$ | Hidden layers (zero-centered) |
| **ReLU** | $\max(0, z)$ | $1 	ext{ if } z > 0 	ext{ else } 0$ | Modern standard for hidden layers |
| **Leaky ReLU** | $\max(lpha z, z)$ ($lpha = 0.01$) | $1 	ext{ if } z > 0 	ext{ else } lpha$ | Prevents "dying ReLU" |
| **Softmax** | $rac{e^{z_i}}{\sum_j e^{z_j}}$ | $\hat{y}_i(\delta_{ij} - \hat{y}_j)$ | Multi-class classification output |

Let's implement each function and its derivative in pure NumPy and plot them side by side.


In [ ]:
# Activation functions and their gradients
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1.0 - np.tanh(z)**2

def relu(z):
    return np.maximum(0.0, z)

def relu_derivative(z):
    return np.where(z > 0, 1.0, 0.0)

def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.1):
    return np.where(z > 0, 1.0, alpha)

z_vals = np.linspace(-6, 6, 400)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

activations = [
    ("Sigmoid", sigmoid(z_vals), sigmoid_derivative(z_vals), "Max Gradient = 0.25"),
    ("Tanh", tanh(z_vals), tanh_derivative(z_vals), "Max Gradient = 1.0 (Zero-centered)"),
    ("ReLU", relu(z_vals), relu_derivative(z_vals), "No saturation for z > 0"),
    ("Leaky ReLU (α=0.1)", leaky_relu(z_vals), leaky_relu_derivative(z_vals), "Non-zero gradient for z < 0")
]

for ax, (name, a, da, note) in zip(axes.ravel(), activations):
    ax.plot(z_vals, a, label=f"{name} Activation $g(z)$", color="dodgerblue", lw=2.5)
    ax.plot(z_vals, da, label=f"Derivative $g'(z)$", color="darkorange", lw=2, linestyle="--")
    ax.axhline(0, color="gray", lw=0.8, alpha=0.7)
    ax.axvline(0, color="gray", lw=0.8, alpha=0.7)
    ax.set_title(f"{name} ({note})", fontsize=12, fontweight="bold")
    ax.set_ylim(-1.2, 2.5)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


### 2.4 The Vanishing Gradient Problem
Look carefully at the derivative plots above:
- For **Sigmoid**, the maximum derivative is only $0.25$ (at $z=0$). For $|z| > 4$, $g'(z) pprox 0$.
- By the chain rule, gradients backpropagated through $L$ sigmoid layers decay exponentially by a factor of $(0.25)^L$. By layer 4, gradients are scaled down by $(0.25)^4 pprox 0.0039$, stalling learning in early layers.
- **ReLU** maintains a constant derivative of $1.0$ for all positive values, completely eliminating saturation and enabling the training of very deep networks.


---
## 3. Loss Functions & Optimization Algorithms (SGD vs. Adam)

### 3.1 Loss Functions
A loss function measures the discrepancy between predicted probability $\hat{y}$ and true label $y \in \{0, 1\}$.

#### Binary Cross-Entropy (BCE / Log-Loss):
For $m$ samples:
$$J(W, b) = -rac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \log(\hat{y}^{(i)} + \epsilon) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)} + \epsilon) ight]$$
where $\epsilon = 10^{-15}$ prevents $\log(0)$ numerical undefined behavior.

The gradient with respect to output activation $\hat{y}$ combined with the sigmoid derivative yields a remarkably clean gradient:
$$rac{\partial J}{\partial z^{[L]}} = \hat{y} - y$$

Let's implement Binary Cross-Entropy and Mean Squared Error in NumPy:


In [ ]:
def binary_cross_entropy(y_true, y_pred, eps=1e-15):
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    m = y_true.shape[0]
    loss = - (1.0 / m) * np.sum(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))
    return float(loss)

def mean_squared_error(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

# Quick verification test
y_dummy = np.array([[1], [0], [1], [0]])
y_pred_good = np.array([[0.95], [0.05], [0.90], [0.10]])
y_pred_bad  = np.array([[0.10], [0.80], [0.20], [0.90]])

print(f"BCE Loss (Confident & Correct): {binary_cross_entropy(y_dummy, y_pred_good):.4f}")
print(f"BCE Loss (Confident & Wrong):   {binary_cross_entropy(y_dummy, y_pred_bad):.4f}")


### 3.2 Optimization Algorithms: From SGD to Adam
An optimizer updates weights in the direction that minimizes loss $J(\mathbf{w})$.

#### 1. Stochastic Gradient Descent (SGD)
$$\mathbf{w}_{t} = \mathbf{w}_{t-1} - lpha 
abla_{\mathbf{w}} J$$
- Simple, but prone to oscillations in ravines and slow progression along plateaus.

#### 2. SGD with Momentum
Maintains an exponential moving average of past gradients (like a heavy ball rolling down a hill):
$$\mathbf{v}_t = eta \mathbf{v}_{t-1} + (1 - eta) 
abla_{\mathbf{w}} J$$
$$\mathbf{w}_t = \mathbf{w}_{t-1} - lpha \mathbf{v}_t$$
- Damps oscillations in perpendicular directions and accelerates down gentle slopes. Typically $eta = 0.9$.

#### 3. Adam (Adaptive Moment Estimation)
Combines **Momentum** (1st moment, mean of gradients) and **RMSprop** (2nd moment, uncentered variance of gradients) with **bias correction**:
1. First moment (gradient mean):
   $$m_t = eta_1 m_{t-1} + (1 - eta_1) g_t$$
2. Second moment (uncentered variance):
   $$v_t = eta_2 v_{t-1} + (1 - eta_2) g_t^2$$
3. Bias corrections (compensating for initial zero initialization):
   $$\hat{m}_t = rac{m_t}{1 - eta_1^t}, \quad \hat{v}_t = rac{v_t}{1 - eta_2^t}$$
4. Parameter update:
   $$\mathbf{w}_t = \mathbf{w}_{t-1} - rac{lpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$
*(Standard default parameters: $lpha = 0.001$, $eta_1 = 0.9$, $eta_2 = 0.999$, $\epsilon = 10^{-8}$)*

Let's visualize their optimization paths across a non-convex 2D "ravine" loss surface: $f(x, y) = 0.1 x^2 + 2 y^2$ (a sharp elliptical bowl).


In [ ]:
def loss_surface(x, y):
    return 0.1 * x**2 + 2.0 * y**2

def grad_loss_surface(x, y):
    return np.array([0.2 * x, 4.0 * y])

# Optimizer path simulation
def run_optimizer(opt_name, start_pos=(9.0, 3.5), n_steps=60, lr=0.15):
    trajectory = [np.array(start_pos, dtype=float)]
    w = np.array(start_pos, dtype=float)
    
    if opt_name == "SGD":
        for _ in range(n_steps):
            g = grad_loss_surface(w[0], w[1])
            w = w - lr * g
            trajectory.append(w.copy())
            
    elif opt_name == "SGD+Momentum":
        v = np.zeros(2)
        beta = 0.9
        for _ in range(n_steps):
            g = grad_loss_surface(w[0], w[1])
            v = beta * v + (1 - beta) * g
            w = w - lr * v
            trajectory.append(w.copy())
            
    elif opt_name == "Adam":
        m = np.zeros(2)
        v = np.zeros(2)
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        for t in range(1, n_steps + 1):
            g = grad_loss_surface(w[0], w[1])
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * (g ** 2)
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            w = w - (lr / (np.sqrt(v_hat) + eps)) * m_hat
            trajectory.append(w.copy())
            
    return np.array(trajectory)

traj_sgd = run_optimizer("SGD", lr=0.35)
traj_mom = run_optimizer("SGD+Momentum", lr=0.35)
traj_adam = run_optimizer("Adam", lr=0.6)

# Plot loss contours
x_grid = np.linspace(-10, 10, 200)
y_grid = np.linspace(-4, 4, 200)
X_g, Y_g = np.meshgrid(x_grid, y_grid)
Z_g = loss_surface(X_g, Y_g)

plt.figure(figsize=(10, 6))
plt.contour(X_g, Y_g, Z_g, levels=np.logspace(-1, 2.5, 25), cmap="magma", alpha=0.6)
plt.plot(traj_sgd[:, 0], traj_sgd[:, 1], "o--", color="crimson", label="SGD (oscillates in valley)", lw=1.5, ms=4)
plt.plot(traj_mom[:, 0], traj_mom[:, 1], "s-", color="dodgerblue", label="SGD + Momentum (smoother trajectory)", lw=1.5, ms=4)
plt.plot(traj_adam[:, 0], traj_adam[:, 1], "*-", color="lime", label="Adam (fast adaptive convergence)", lw=2, ms=6)

plt.plot(0, 0, "rX", ms=14, label="Global Minimum (0, 0)")
plt.title("Optimization Trajectories on an Ill-Conditioned Ravine", fontsize=13, fontweight="bold")
plt.xlabel("Parameter $w_1$ (gentle curvature)")
plt.ylabel("Parameter $w_2$ (steep curvature)")
plt.legend(loc="upper left")
plt.show()


**Observation**:
- Standard **SGD** wildly oscillates along the steep vertical axis while making painfully slow progress on the horizontal axis.
- **Momentum** cancels out vertical oscillations and drives forward along the valley floor.
- **Adam** dynamically adapts individual learning rates for each parameter, rapidly homing in on the minimum.

---

## 4. Forward & Backward Propagation Mechanics

To implement a deep neural network from scratch in NumPy, we must understand the vectorized mathematics of gradient flow.

### 4.1 Computational Graph for Layer $l$
For a dataset mini-batch with $m$ samples, where layer $l-1$ has activations $A^{[l-1]} \in \mathbb{R}^{m 	imes n_{l-1}}$:

```
    A^[l-1] ----> [ Z^[l] = A^[l-1] * W^[l] + b^[l] ] ----> [ A^[l] = g(Z^[l]) ] ----> Loss J
                         ^                 ^
                       W^[l]             b^[l]
```

### 4.2 Forward Pass Equations
1. **Linear Pre-activation**:
   $$Z^{[l]} = A^{[l-1]} W^{[l]} + b^{[l]}$$
   *(Dimensions: $(m 	imes n_{l-1}) 	imes (n_{l-1} 	imes n_l) + (1 	imes n_l) = (m 	imes n_l)$)*
2. **Post-activation**:
   $$A^{[l]} = g^{[l]}(Z^{[l]})$$

### 4.3 Backward Pass (Backpropagation) Equations
Using the chain rule, we compute derivatives in reverse:
1. **Output layer error** (for Binary Cross-Entropy with Sigmoid output):
   $$dZ^{[L]} = A^{[L]} - Y$$
2. **Hidden layer error** (propagating upstream from layer $l+1$):
   $$dZ^{[l]} = \left( dZ^{[l+1]} (W^{[l+1]})^T ight) \odot {g^{[l]}}'(Z^{[l]})$$
   where $\odot$ is element-wise Hadamard product.
3. **Parameter gradients**:
   $$dW^{[l]} = rac{1}{m} (A^{[l-1]})^T dZ^{[l]}$$
   $$db^{[l]} = rac{1}{m} \sum_{i=1}^m dZ^{[l]} \quad 	ext{(row sum vector)}$$


---
## 5. Building a Modular Neural Network from Scratch in NumPy

Let's now assemble an object-oriented, modular neural network.

We will build:
1. `DenseLayer`: Handles weight initialization (**He / Kaiming initialization** for ReLU, **Xavier / Glorot** for Tanh/Sigmoid), linear computation, caching, and backprop.
2. `ModularNeuralNetwork`: Handles layer management, forward passes, loss calculation, backward pass chain-rule orchestration, and parameter updates via **SGD** or **Adam**.

Let's inspect the code implementation:


In [ ]:
class DenseLayer:
    def __init__(self, in_features: int, out_features: int, activation: str = "relu"):
        self.in_features = in_features
        self.out_features = out_features
        self.activation = activation.lower()
        
        # Proper Weight Initialization:
        # He / Kaiming init for ReLU / Leaky ReLU
        # Xavier / Glorot init for Sigmoid / Tanh
        if self.activation in ["relu", "leaky_relu"]:
            limit = np.sqrt(2.0 / in_features)
        else:
            limit = np.sqrt(1.0 / in_features)
            
        self.W = np.random.randn(in_features, out_features) * limit
        self.b = np.zeros((1, out_features))
        
        # Gradients cache
        self.dW = None
        self.db = None
        
        # Forward pass cache
        self.A_prev = None
        self.Z = None
        
        # Adam optimizer state
        self.m_W, self.v_W = np.zeros_like(self.W), np.zeros_like(self.W)
        self.m_b, self.v_b = np.zeros_like(self.b), np.zeros_like(self.b)

    def forward(self, A_prev: np.ndarray) -> np.ndarray:
        self.A_prev = A_prev
        self.Z = np.dot(A_prev, self.W) + self.b
        
        if self.activation == "relu":
            return relu(self.Z)
        elif self.activation == "leaky_relu":
            return leaky_relu(self.Z)
        elif self.activation == "tanh":
            return tanh(self.Z)
        elif self.activation == "sigmoid":
            return sigmoid(self.Z)
        elif self.activation == "linear":
            return self.Z
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")

    def backward(self, dA: np.ndarray) -> np.ndarray:
        m = self.A_prev.shape[0]
        
        # Compute dZ = dA * g'(Z)
        if self.activation == "relu":
            dZ = dA * relu_derivative(self.Z)
        elif self.activation == "leaky_relu":
            dZ = dA * leaky_relu_derivative(self.Z)
        elif self.activation == "tanh":
            dZ = dA * tanh_derivative(self.Z)
        elif self.activation == "sigmoid":
            dZ = dA * sigmoid_derivative(self.Z)
        elif self.activation == "linear":
            dZ = dA
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")
            
        # Parameter gradients
        self.dW = (1.0 / m) * np.dot(self.A_prev.T, dZ)
        self.db = (1.0 / m) * np.sum(dZ, axis=0, keepdims=True)
        
        # Gradient with respect to inputs (propagated upstream to previous layer)
        dA_prev = np.dot(dZ, self.W.T)
        return dA_prev


### 5.2 The Complete Neural Network Class


In [ ]:
class ModularNeuralNetwork:
    def __init__(self, optimizer: str = "adam", learning_rate: float = 0.01):
        self.layers = []
        self.optimizer = optimizer.lower()
        self.lr = learning_rate
        self.t = 0  # Timestep for Adam bias-correction
        self.loss_history = []
        self.acc_history = []
        
    def add_layer(self, layer: DenseLayer):
        self.layers.append(layer)
        return self
        
    def forward(self, X: np.ndarray) -> np.ndarray:
        A = X
        for layer in self.layers:
            A = layer.forward(A)
        return A

    def backward(self, y_true: np.ndarray, y_pred: np.ndarray):
        m = y_true.shape[0]
        # For Binary Cross Entropy with Sigmoid output, dZ_last = (A - Y)
        # Note: self.layers[-1].backward takes dA. For sigmoid output + BCE loss:
        # dA_last = (A - Y) / (A * (1 - A) + eps)
        # Since layer.backward multiplies dA_last * sigmoid_derivative = (A - Y) exactly!
        eps = 1e-15
        y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
        dA = - (y_true / y_pred_clipped) + ((1.0 - y_true) / (1.0 - y_pred_clipped))
        
        for layer in reversed(self.layers):
            dA = layer.backward(dA)

    def step(self):
        self.t += 1
        for layer in self.layers:
            if self.optimizer == "sgd":
                layer.W -= self.lr * layer.dW
                layer.b -= self.lr * layer.db
                
            elif self.optimizer == "adam":
                beta1, beta2, eps = 0.9, 0.999, 1e-8
                
                # Update 1st and 2nd moment for W
                layer.m_W = beta1 * layer.m_W + (1 - beta1) * layer.dW
                layer.v_W = beta2 * layer.v_W + (1 - beta2) * (layer.dW ** 2)
                m_W_hat = layer.m_W / (1.0 - beta1 ** self.t)
                v_W_hat = layer.v_W / (1.0 - beta2 ** self.t)
                layer.W -= (self.lr / (np.sqrt(v_W_hat) + eps)) * m_W_hat
                
                # Update 1st and 2nd moment for b
                layer.m_b = beta1 * layer.m_b + (1 - beta1) * layer.db
                layer.v_b = beta2 * layer.v_b + (1 - beta2) * (layer.db ** 2)
                m_b_hat = layer.m_b / (1.0 - beta1 ** self.t)
                v_b_hat = layer.v_b / (1.0 - beta2 ** self.t)
                layer.b -= (self.lr / (np.sqrt(v_b_hat) + eps)) * m_b_hat

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 500, verbose: int = 100):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
            
        for epoch in range(1, epochs + 1):
            # 1. Forward propagation
            y_pred = self.forward(X)
            
            # 2. Compute Loss & Accuracy
            loss = binary_cross_entropy(y, y_pred)
            preds_binary = (y_pred >= 0.5).astype(int)
            acc = float(np.mean(preds_binary == y))
            
            self.loss_history.append(loss)
            self.acc_history.append(acc)
            
            # 3. Backpropagation
            self.backward(y, y_pred)
            
            # 4. Parameter update
            self.step()
            
            if verbose and (epoch % verbose == 0 or epoch == epochs):
                print(f"Epoch {epoch:4d}/{epochs} | Loss: {loss:.5f} | Accuracy: {acc:.2%}")
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.forward(X)
        
    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        probas = self.predict_proba(X)
        return (probas >= threshold).astype(int)


---
## 6. Training the Scratch Neural Net on Non-Linear Moons Dataset

Let's test our pure NumPy neural network on the famous **Two Moons** dataset, which is completely non-linearly separable.

We will configure an architecture with:
- **Input Layer**: 2 features $(x_1, x_2)$
- **Hidden Layer 1**: 16 neurons with **ReLU**
- **Hidden Layer 2**: 8 neurons with **ReLU**
- **Output Layer**: 1 neuron with **Sigmoid** activation
- **Optimizer**: **Adam** ($	ext{lr} = 0.02$)


In [ ]:
# Generate non-linear dataset
X_moons, y_moons = make_moons(n_samples=1000, noise=0.20, random_state=RANDOM_STATE)
y_moons = y_moons.reshape(-1, 1)

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=RANDOM_STATE, stratify=y_moons
)

# Scale inputs
scaler_m = StandardScaler()
X_m_train_s = scaler_m.fit_transform(X_m_train)
X_m_test_s  = scaler_m.transform(X_m_test)

# Build custom NumPy network
nn_scratch = ModularNeuralNetwork(optimizer="adam", learning_rate=0.02)
nn_scratch.add_layer(DenseLayer(in_features=2, out_features=16, activation="relu"))
nn_scratch.add_layer(DenseLayer(in_features=16, out_features=8, activation="relu"))
nn_scratch.add_layer(DenseLayer(in_features=8, out_features=1, activation="sigmoid"))

print("Starting training of pure NumPy Neural Network...")
nn_scratch.fit(X_m_train_s, y_m_train, epochs=400, verbose=80)

# Evaluate on unseen test set
y_m_pred = nn_scratch.predict(X_m_test_s)
test_acc = accuracy_score(y_m_test, y_m_pred)
print(f"\nFinal Test Accuracy on Unseen Data: {test_acc:.2%}")


### 6.1 Visualizing the Learned Non-Linear Decision Boundary
Let's plot the decision boundary learned by our NumPy neural network against the raw data points:


In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = model.predict_proba(grid).reshape(xx.shape)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Decision Boundary contour
    contour = axes[0].contourf(xx, yy, probs, levels=25, cmap="coolwarm", alpha=0.85)
    axes[0].contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=2.5, linestyles="--")
    scatter = axes[0].scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", edgecolors="k", s=30)
    axes[0].set_title(f"{title} (Contour & Cutoff)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Feature 1 (Standardized)")
    axes[0].set_ylabel("Feature 2 (Standardized)")
    fig.colorbar(contour, ax=axes[0], label="Predicted Probability $\hat{y}$")
    
    # Subplot 2: Learning Curves
    epochs = range(1, len(model.loss_history) + 1)
    ax2 = axes[1]
    ax2.plot(epochs, model.loss_history, color="crimson", lw=2, label="BCE Loss")
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Loss", color="crimson")
    ax2.tick_params(axis="y", labelcolor="crimson")
    
    ax2_right = ax2.twinx()
    ax2_right.plot(epochs, model.acc_history, color="navy", lw=2, linestyle=":", label="Training Accuracy")
    ax2_right.set_ylabel("Accuracy", color="navy")
    ax2_right.tick_params(axis="y", labelcolor="navy")
    ax2.set_title("Training Loss & Accuracy Curves", fontsize=12, fontweight="bold")
    
    plt.tight_layout()
    plt.show()

plot_decision_boundary(nn_scratch, X_m_test_s, y_m_test, title="NumPy Neural Net Decision Surface")


---
## 7. Full Integration: Serializing & Deploying the Scratch Neural Network

Can our scratch-built NumPy model be serialized and deployed with `joblib` just like a scikit-learn model? **Yes!** Because our classes are standard Python objects holding NumPy arrays, `joblib` and `pickle` handle them seamlessly.

Let's save the model and preprocessor, reload them, and run production verification tests:


In [ ]:
scratch_model_path = "models/scratch_neural_net.joblib"
scratch_preprocessor_path = "models/scratch_scaler.joblib"

# Save trained scratch model and scaler
joblib.dump(nn_scratch, scratch_model_path)
joblib.dump(scaler_m, scratch_preprocessor_path)

print(f"Scratch model saved to: {scratch_model_path} ({os.path.getsize(scratch_model_path) / 1024:.2f} KB)")
print(f"Scaler saved to:        {scratch_preprocessor_path} ({os.path.getsize(scratch_preprocessor_path) / 1024:.2f} KB)")

# Reload into fresh variables
deployed_nn = joblib.load(scratch_model_path)
deployed_scaler = joblib.load(scratch_preprocessor_path)

# Verify bitwise parity on test set
pre_save_preds = nn_scratch.predict_proba(X_m_test_s)
post_load_preds = deployed_nn.predict_proba(X_m_test_s)

assert np.allclose(pre_save_preds, post_load_preds), "Reloaded model outputs do not match original!"
print("Deployment Parity Test: PASSED (100% Bitwise Match)!")

# Mock real-time streaming inference
def score_raw_observation(feature_1: float, feature_2: float):
    """Simulates live microservice inference on new streaming input."""
    raw_point = np.array([[feature_1, feature_2]])
    scaled_point = deployed_scaler.transform(raw_point)
    prob = float(deployed_nn.predict_proba(scaled_point)[0, 0])
    label = int(prob >= 0.5)
    return {
        "raw_features": {"x1": feature_1, "x2": feature_2},
        "predicted_label": label,
        "class_name": "Moon Class A" if label == 0 else "Moon Class B",
        "confidence": round(prob if label == 1 else (1.0 - prob), 4)
    }

print("\nSample Live Predictions:")
print(score_raw_observation(1.2, -0.4))
print(score_raw_observation(-0.8, 0.9))


---
## 8. Summary & Key Takeaways

### Conceptual Recap
| Topic | Core Concept | Practical Guidance |
| :--- | :--- | :--- |
| **Model Deployment** | Model persistence via `pickle` / `joblib`. | Bundle feature scalers and models together into atomic pipelines. Prefer `joblib` for large NumPy models. |
| **Perceptron Limits** | Single linear boundary $\mathbf{w}^T \mathbf{x} + b$. | Cannot solve XOR. Needs multi-layer architectures + non-linear activations. |
| **Activations** | Sigmoid, Tanh, ReLU, Leaky ReLU. | Use **ReLU** / **Leaky ReLU** for hidden layers to avoid vanishing gradients; **Sigmoid** for binary classification output. |
| **Loss & Gradients** | Binary Cross-Entropy with Sigmoid output. | Yields the clean gradient $dZ = \hat{y} - y$. |
| **Optimizers** | SGD, Momentum, Adam. | **Adam** adapts individual learning rates with bias-corrected 1st and 2nd moments, outperforming vanilla SGD on ravines. |
| **Scratch Neural Net** | Forward, Loss, Backward, Step. | Vectorized batch operations in NumPy compute exact gradients via the matrix chain rule. |

---

### Student Exercises & Challenges
To reinforce your mastery, try the following challenges:
1. **Multiclass Softmax Extension**: Modify `DenseLayer` and `ModularNeuralNetwork` to support Softmax activation $\hat{y}_k = rac{e^{z_k}}{\sum_j e^{z_j}}$ and Categorical Cross-Entropy loss for $K > 2$ classes. Test it on the Iris or Digits dataset.
2. **L2 Regularization (Weight Decay)**: Add an L2 penalty $rac{\lambda}{2m} \sum ||W^{[l]}||_2^2$ to the loss function and update the weight gradient: $dW^{[l]} \leftarrow dW^{[l]} + rac{\lambda}{m} W^{[l]}$.
3. **Mini-batch Training**: Add mini-batch slicing (e.g. `batch_size=32`) to the `fit` method instead of full-batch training, and observe the stochastic behavior of the training loss curve.
